# Baseline: Differential Expression Analysis per Cluster

This tutorial performs differential expression (DE) analysis on clusters obtained from Louvain clustering and extracts cluster-specific marker genes. These marker genes serve as a baseline "topic" representation for comparison with scE2TM's learned topics.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

# ========== Path Configuration ==========
# Get current notebook directory and set project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # Go up one level from baseline/ to scE2TM_github/
DATA_DIR = PROJECT_ROOT / 'data'

DATASET_NAME = 'Wang'

# File paths
expression_file = DATA_DIR / f'{DATASET_NAME}_HIGHPRE.csv'
label_file = DATA_DIR / f'{DATASET_NAME}_cell_anno.csv'

print("="*60)
print("Baseline: Louvain Clustering on PCA-reduced Data")
print(f"Dataset: {DATASET_NAME}")
print("="*60)

# ========== Load data ==========
print("\n1. Loading data...")
expression_df = pd.read_csv(expression_file, sep=',', index_col=0)
adata = sc.AnnData(expression_df)
print(f"   Expression data shape: {adata.shape}")

# Load labels
label_df = pd.read_csv(label_file, sep=',', index_col=0)
adata.obs['cell_type'] = list(label_df.iloc[:, 0])
true_labels = adata.obs['cell_type'].values
n_cell_types = len(np.unique(true_labels))
print(f"   True labels loaded: {n_cell_types} cell types")

# ========== PCA and Neighbor graph ==========
print("\n2. PCA and neighbor graph construction...")
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata, use_rep='X_pca')
print(f"   PCA completed.")

# ========== Find optimal clustering resolution ==========
print("\n3. Finding optimal clustering resolution by maximizing ARI...")

max_resolution = 2
min_resolution = 0

ari_scores = []
resolutions = []

for x in range(int(min_resolution * 10), int(max_resolution * 10) + 1):
    resolution = x / 10.0
    sc.tl.louvain(adata, resolution=resolution, random_state=0, key_added='louvain')
    ari = adjusted_rand_score(adata.obs['cell_type'], adata.obs['louvain'])
    ari_scores.append(ari)
    resolutions.append(resolution)

# Find best resolution
best_idx = np.argmax(ari_scores)
best_resolution = resolutions[best_idx]
best_ari = ari_scores[best_idx]

print(f"   Best resolution: {best_resolution:.1f} (ARI = {best_ari:.4f})")

# Run final clustering with best resolution
sc.tl.louvain(adata, resolution=best_resolution, random_state=0, key_added='louvain')

n_clusters = len(adata.obs['louvain'].unique())
print(f"   Number of clusters: {n_clusters}")

# ========== UMAP visualization ==========
print("\n4. Generating UMAP visualization...")
sc.tl.umap(adata)

# Plot UMAP with both true labels and clustering results
sc.pl.umap(
    adata,
    color=["cell_type", "louvain"],
    wspace=0.3,
    frameon=False,
    title=["True Cell Types", f"Louvain Clusters (n={n_clusters})"]
)

# ========== Evaluation metrics ==========
print("\n5. Evaluation metrics...")

ari = adjusted_rand_score(adata.obs['cell_type'], adata.obs['louvain'])
nmi = adjusted_mutual_info_score(adata.obs['cell_type'], adata.obs['louvain'])

print(f"\n   Adjusted Rand Index (ARI): {ari:.4f}")
print(f"   Normalized Mutual Info (NMI): {nmi:.4f}")
print(f"   Number of clusters: {n_clusters}")
print(f"   Number of true cell types: {n_cell_types}")
print(f"   Optimal Louvain resolution: {best_resolution:.1f}")

# ========== Summary ==========
print("\n" + "="*60)
print("Summary")
print("="*60)
print(f"Dataset: {DATASET_NAME}")
print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")
print(f"Number of clusters: {n_clusters}")

print("\n" + "="*60)
print("Baseline completed!")
print("="*60)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_topic_capture(adata, type_key="cell_type", topic_key="louvain"):
    """
    Evaluate how well each cell type is captured by the best matching topic.
    
    For each cell type:
        1. Find the topic (cluster) that contains the most cells of this type
        2. Treat this as the "representative topic" for that cell type
        3. Compute recall, precision, F1, and purity
    
    Parameters
    ----------
    adata : AnnData
        AnnData object containing cell type annotations and cluster assignments
    type_key : str
        Key in adata.obs for true cell type labels
    topic_key : str
        Key in adata.obs for cluster/topic assignments
    
    Returns
    -------
    pd.DataFrame
        DataFrame with evaluation metrics for each cell type
    """
    
    # Create DataFrame with relevant columns
    df = adata.obs[[type_key, topic_key]].copy()
    
    # Convert to string for consistent handling
    df[type_key] = df[type_key].astype(str)
    df[topic_key] = df[topic_key].astype(str)
    
    # Exclude cells with cluster -1 (noise/outliers) if present
    if '-1' in df[topic_key].values:
        print(f"   Warning: Found cells with topic '-1' (outliers). Excluding from analysis.")
        df = df[df[topic_key] != '-1']
    
    results = []
    cell_types = df[type_key].unique()
    
    print(f"\n   Analyzing {len(cell_types)} cell types...")
    
    for ct in cell_types:
        # Subset for current cell type
        sub = df[df[type_key] == ct]
        
        # Count distribution of topics within this cell type
        topic_counts = sub[topic_key].value_counts()
        
        # Find best matching topic (most frequent)
        best_topic = topic_counts.index[0]
        
        # Binary classification: is this cell type?
        y_true = (df[type_key] == ct).astype(int)
        y_pred = (df[topic_key] == best_topic).astype(int)
        
        # Calculate metrics
        recall = recall_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        
        # Purity: proportion of cells in this topic that belong to this cell type
        topic_subset = df[df[topic_key] == best_topic]
        purity = (topic_subset[type_key] == ct).mean()
        
        results.append({
            "cell_type": ct,
            "best_topic": best_topic,
            "n_cells": len(sub),
            "n_cells_in_topic": len(topic_subset),
            "topic_purity": round(purity, 4),
            "recall": round(recall, 4),
            "precision": round(precision, 4),
            "f1": round(f1, 4)
        })
    
    result_df = pd.DataFrame(results).sort_values("f1", ascending=False)
    return result_df


# ========== Run evaluation ==========
print("="*60)
print("Topic Capture Evaluation per Cell Type")
print("="*60)

# Use Louvain clusters as topics
adata.obs["topic_ass"] = adata.obs['louvain']

# Evaluate
result_df = evaluate_topic_capture(adata, type_key="cell_type", topic_key="louvain")

# Display results
print("\n" + "="*60)
print("Results: Best matching topic for each cell type")
print("="*60)
print(result_df.to_string(index=False))

# ========== Summary statistics ==========
print("\n" + "="*60)
print("Summary Statistics")
print("="*60)
print(f"Average F1 score: {result_df['f1'].mean():.4f}")
print(f"Average recall: {result_df['recall'].mean():.4f}")
print(f"Average precision: {result_df['precision'].mean():.4f}")
print(f"Average topic purity: {result_df['topic_purity'].mean():.4f}")

# Find best and worst performing cell types
best_idx = result_df['f1'].idxmax()
worst_idx = result_df['f1'].idxmin()

print(f"\nBest captured cell type: {result_df.loc[best_idx, 'cell_type']} (F1={result_df.loc[best_idx, 'f1']:.4f})")
print(f"Worst captured cell type: {result_df.loc[worst_idx, 'cell_type']} (F1={result_df.loc[worst_idx, 'f1']:.4f})")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ========== Set plot parameters ==========
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']
rcParams['font.size'] = 8
rcParams['figure.dpi'] = 300

# ========== Data info ==========
print("="*60)
print("Rare Cell Types Visualization")
print("="*60)

print(f"adata shape: {adata.shape}")
print(f"Cell types: {adata.obs['cell_type'].unique()}")
print(f"Topic assignments: {adata.obs['topic_ass'].unique()}")

# ========== Find rarest cell types ==========
cell_type_counts = adata.obs['cell_type'].value_counts().sort_values()
print("\nCell type counts (sorted):")
print(cell_type_counts)

# Get two rarest cell types
rarest_cell_type = cell_type_counts.index[0]
second_rarest_cell_type = cell_type_counts.index[1]

# Find their best matching topics from result_df
rarest_topic = int(result_df[result_df['cell_type'] == rarest_cell_type]['best_topic'].values[0])
second_rarest_topic = int(result_df[result_df['cell_type'] == second_rarest_cell_type]['best_topic'].values[0])

print(f"\nRarest cell type: {rarest_cell_type}")
print(f"  Number of cells: {cell_type_counts[rarest_cell_type]}")
print(f"  Best matching topic: {rarest_topic}")
print(f"  Cells in this topic: {(adata.obs['topic_ass'].astype(int) == rarest_topic).sum()}")

print(f"\nSecond rarest cell type: {second_rarest_cell_type}")
print(f"  Number of cells: {cell_type_counts[second_rarest_cell_type]}")
print(f"  Best matching topic: {second_rarest_topic}")
print(f"  Cells in this topic: {(adata.obs['topic_ass'].astype(int) == second_rarest_topic).sum()}")

# ========== Helper function ==========
def create_highlight_column(adata, highlight_values, column_name, highlight_color=None):
    """Create highlight column, set non-highlight values to NaN"""
    new_col = adata.obs[column_name].copy()
    mask = ~adata.obs[column_name].isin(highlight_values)
    new_col[mask] = np.nan
    return new_col

# ========== Plot for each rare cell type ==========
for cell_type, best_topic in zip([rarest_cell_type, second_rarest_cell_type], 
                                   [rarest_topic, second_rarest_topic]):
    print(f"\nPlotting {cell_type}...")
    
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
    
    # Left plot: Highlight the rare cell type
    highlight_cell = create_highlight_column(adata, [cell_type], 'cell_type')
    adata.obs[f'highlight_{cell_type}'] = highlight_cell
    
    sc.pl.umap(
        adata,
        color=f'highlight_{cell_type}',
        ax=axes[0],
        title=cell_type,
        frameon=False,
        show=False,
        size=50,
        legend_loc='none',
        palette=['#E64B35'],  # Red color
        na_color='lightgray'
    )
    
    # Right plot: Highlight the best matching topic
    topic_labels = np.full(len(adata), np.nan, dtype=object)
    topic_mask = adata.obs['topic_ass'].astype(int) == best_topic
    topic_labels[topic_mask] = f'Topic {best_topic}'
    adata.obs[f'topic_{best_topic}_highlight'] = topic_labels
    
    sc.pl.umap(
        adata,
        color=f'topic_{best_topic}_highlight',
        ax=axes[1],
        title=f'Topic {best_topic}',
        frameon=False,
        show=False,
        size=50,
        legend_loc='none',
        palette=['#4DBBD5'],  # Blue color
        na_color='lightgray'
    )
    
    # Formatting
    axes[0].set_title(cell_type, fontsize=12)
    axes[1].set_title(f'Topic {best_topic}', fontsize=12)
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    
    plt.tight_layout()
    plt.show()
    
    # Clean up temporary columns
    del adata.obs[f'highlight_{cell_type}']
    del adata.obs[f'topic_{best_topic}_highlight']

print("\n" + "="*60)
print("Visualization completed!")
print("="*60)

In [ ]:

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ========== Set plot parameters ==========
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']
rcParams['font.size'] = 8
rcParams['figure.dpi'] = 300

# ========== Differential expression analysis ==========
print("="*60)
print("Differential Expression Analysis per Cluster")
print("="*60)

# Check if 'louvain' column exists
if 'louvain' not in adata.obs.columns:
    print("Error: 'louvain' column not found. Please run clustering first.")
else:
    n_clusters = len(adata.obs['louvain'].unique())
    print(f"Number of clusters: {n_clusters}")

# ========== Run rank_genes_groups ==========
print("\n1. Computing differentially expressed genes for each cluster...")

# Make sure data is properly normalized for DE analysis
if 'raw' in adata.layers:
    sc.tl.rank_genes_groups(
        adata, 
        groupby='louvain', 
        layer='raw',
        method='t-test', 
        n_genes=20,
        pts=True
    )
else:
    sc.tl.rank_genes_groups(
        adata, 
        groupby='louvain', 
        method='t-test', 
        n_genes=20,
        pts=True
    )

print("   rank_genes_groups completed!")

# ========== Verify results ==========
print("\n2. Verifying results...")
print(f"   'rank_genes_groups' in uns: {'rank_genes_groups' in adata.uns}")

if 'rank_genes_groups' in adata.uns:
    print(f"   Keys in rank_genes_groups: {adata.uns['rank_genes_groups'].keys()}")
    
    # Direct access to results
    rank_dict = adata.uns['rank_genes_groups']
    names = rank_dict['names']
    scores = rank_dict['scores']
    
    # Get cluster names
    cluster_names = names.dtype.names
    print(f"   Cluster names: {cluster_names}")
    
    # ========== Display top genes per cluster ==========
    print("\n3. Top differentially expressed genes per cluster:")
    
    for cluster_name in cluster_names:
        print(f"\n   Cluster {cluster_name}:")
        top_genes = names[cluster_name][:10]
        top_scores = scores[cluster_name][:10]
        for i, (gene, score) in enumerate(zip(top_genes, top_scores)):
            print(f"      {i+1}. {gene} (score: {score:.4f})")
    
    # ========== Extract cluster gene sets ==========
    print("\n4. Extracting cluster gene sets...")
    
    cluster_genes = {}
    for cluster_name in cluster_names:
        top_genes = names[cluster_name][:20].tolist()
        # Convert cluster_name to int for consistency
        cluster_genes[int(cluster_name)] = top_genes
        print(f"   Cluster {cluster_name}: {len(top_genes)} genes extracted")
    
    # ========== Create binary topic-gene matrix ==========
    print("\n5. Creating binary topic-gene matrix...")
    
    # Get all genes
    all_genes = adata.var_names.tolist()
    cluster_ids = sorted([int(c) for c in cluster_names])
    
    # Initialize binary matrix (clusters x genes)
    topic_gene_matrix = pd.DataFrame(0, index=cluster_ids, columns=all_genes, dtype=int)
    
    # Fill matrix: 1 if gene is in top 20 for that cluster
    for cluster, genes in cluster_genes.items():
        for gene in genes:
            if gene in topic_gene_matrix.columns:
                topic_gene_matrix.loc[cluster, gene] = 1
    
    print(f"   Binary topic-gene matrix shape: {topic_gene_matrix.shape}")
    print(f"   Number of non-zero entries: {topic_gene_matrix.sum().sum()}")
    
    # ========== Marker genes for rare cell types ==========
    print("\n6. Marker genes for rare cell types:")
    
    # Get cell type counts
    cell_type_counts = adata.obs['cell_type'].value_counts().sort_values()
    rarest_cell_type = cell_type_counts.index[0]
    second_rarest_cell_type = cell_type_counts.index[1]
    
    # Find which clusters these rare cells belong to
    rare_clusters = {}
    for ct in [rarest_cell_type, second_rarest_cell_type]:
        ct_mask = adata.obs['cell_type'] == ct
        if ct_mask.any():
            # Get dominant cluster as integer
            dominant_cluster = int(adata.obs[ct_mask]['louvain'].mode()[0])
            rare_clusters[ct] = dominant_cluster
            print(f"\n   {ct}:")
            print(f"      Dominant cluster: {dominant_cluster}")
            print(f"      Top 5 marker genes:")
            if dominant_cluster in cluster_genes:
                for i, gene in enumerate(cluster_genes[dominant_cluster][:5]):
                    print(f"         {i+1}. {gene}")
        else:
            print(f"\n   {ct}: No cells found in dataset")
    
    # ========== Summary ==========
    print("\n" + "="*60)
    print("Summary")
    print("="*60)
    print(f"Number of clusters: {len(cluster_names)}")
    print(f"Genes per cluster (top): 20")
    unique_genes = set([g for genes in cluster_genes.values() for g in genes])
    print(f"Total unique marker genes: {len(unique_genes)}")
    
    # ========== Marker gene table for rare cell types ==========
    print("\n" + "="*60)
    print("Marker Genes Summary for Rare Cell Types")
    print("="*60)
    for ct, cluster in rare_clusters.items():
        print(f"\n{ct} (Cluster {cluster}):")
        # Ensure cluster is integer for lookup
        if cluster in cluster_genes:
            genes_str = ', '.join([str(g) for g in cluster_genes[cluster][:10]])
            print(f"   Top 10 marker genes: {genes_str}")
        else:
            print(f"   Cluster {cluster} not found in cluster_genes")
    
else:
    print("   Error: rank_genes_groups not found in adata.uns")

print("\n" + "="*60)
print("Differential expression analysis completed!")
print("="*60)